# Circuit Quest: Power the Grid

Put your team name and colours into the game. Then rotate circuit tiles and light every lamp.

**Order:** set up → change two lines of code → play → open the Life Checkpoint.


## 1. Set up the game

Run the collapsed code cell below. It loads the buttons and game rules. You do not need to edit it.


In [ ]:
import copy

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    import micropip
    print("Preparing the game. This may take a few seconds...")
    await micropip.install("ipywidgets>=8.1,<9")
    import ipywidgets as widgets

from IPython.display import display


# Direction bits: north, east, south, west.
NORTH, EAST, SOUTH, WEST = 1, 2, 4, 8
DIRECTIONS = (NORTH, EAST, SOUTH, WEST)
OPPOSITE = {NORTH: SOUTH, EAST: WEST, SOUTH: NORTH, WEST: EAST}
STEP = {NORTH: (-1, 0), EAST: (0, 1), SOUTH: (1, 0), WEST: (0, -1)}

POWERED_COLOUR = "#b8ff5a"
WIRE_COLOUR = "#35bde0"
BOARD_COLOUR = "#17202a"
LAMP_COLOUR = "#ffd23f"
TEAM_NAME = "Circuit Crew"

GLYPH = {
    0: " ", NORTH: "╵", EAST: "╴", SOUTH: "╷", WEST: "╶",
    NORTH | SOUTH: "│", EAST | WEST: "─",
    NORTH | EAST: "└", EAST | SOUTH: "┌",
    SOUTH | WEST: "┐", WEST | NORTH: "┘",
    NORTH | EAST | SOUTH: "├", EAST | SOUTH | WEST: "┬",
    SOUTH | WEST | NORTH: "┤", WEST | NORTH | EAST: "┴",
    NORTH | EAST | SOUTH | WEST: "┼",
}


def rotate(bits, turns=1):
    for _ in range(turns % 4):
        bits = ((bits << 1) & 15) | ((bits >> 3) & 1)
    return bits


def tile(kind="blank", solved=0, turns=0, locked=False):
    return {"kind": kind, "solved": solved, "turns": turns, "locked": locked}


B = "battery"
W = "wire"
L = "lamp"
X = "blank"

LEVELS = [
    {
        "title": "One turn to power",
        "message": "Rotate the middle wire so the battery can reach the lamp.",
        "board": [
            [tile(B, EAST, locked=True), tile(W, EAST | WEST, turns=1), tile(L, WEST, locked=True)],
            [tile(X), tile(X), tile(X)],
        ],
    },
    {
        "title": "Route around the corner",
        "message": "Build one continuous path. Every wire must meet the next wire.",
        "board": [
            [tile(B, EAST, locked=True), tile(W, EAST | WEST, turns=1), tile(W, SOUTH | WEST, turns=3)],
            [tile(X), tile(X), tile(W, NORTH | SOUTH, turns=1)],
            [tile(L, EAST, locked=True), tile(W, EAST | WEST, turns=1), tile(W, NORTH | WEST, turns=2)],
        ],
    },
    {
        "title": "Power two lamps",
        "message": "Use the branch to send power to both lamps.",
        "board": [
            [tile(X), tile(W, EAST | SOUTH, turns=2), tile(L, WEST, locked=True)],
            [tile(B, EAST, locked=True), tile(W, NORTH | EAST | WEST, turns=2), tile(L, WEST, locked=True)],
            [tile(X), tile(X), tile(X)],
        ],
    },
]

assert len(LEVELS) == 3


class CircuitQuest:
    def __init__(self):
        self.level_index = 0
        self.moves = 0
        self.board = []
        self.buttons = {}
        self.header = widgets.HTML()
        self.instructions = widgets.HTML()
        self.status = widgets.HTML()
        self.reset_button = widgets.Button(description="Reset level", button_style="warning")
        self.hint_button = widgets.Button(description="Hint")
        self.next_button = widgets.Button(description="Next level", button_style="success", disabled=True)
        self.reset_button.on_click(self._reset_clicked)
        self.hint_button.on_click(self._hint_clicked)
        self.next_button.on_click(self._next_clicked)
        self.controls = widgets.HBox(
            [self.reset_button, self.hint_button, self.next_button],
            layout=widgets.Layout(gap="8px"),
        )
        self.grid_holder = widgets.VBox()
        self.view = widgets.VBox(
            [self.header, self.instructions, self.grid_holder, self.status, self.controls],
            layout=widgets.Layout(border="2px solid #35bde0", padding="14px", width="fit-content"),
        )
        self.load_level(0)

    def show(self):
        display(self.view)

    def load_level(self, level_index):
        self.level_index = level_index
        self.moves = 0
        level = LEVELS[level_index]
        self.board = []
        for row in level["board"]:
            new_row = []
            for specification in row:
                current = copy.deepcopy(specification)
                current["bits"] = rotate(current["solved"], current["turns"])
                new_row.append(current)
            self.board.append(new_row)
        self._build_buttons()
        self.refresh()

    def _build_buttons(self):
        self.buttons = {}
        button_list = []
        for row_number, row in enumerate(self.board):
            for column_number, square in enumerate(row):
                button = widgets.Button(
                    description="",
                    tooltip=f"Row {row_number + 1}, column {column_number + 1}",
                    layout=widgets.Layout(width="64px", height="64px", font_size="28px"),
                )
                if square["kind"] == X:
                    button.disabled = True
                elif not square["locked"]:
                    button.on_click(
                        lambda unused, row=row_number, column=column_number: self.turn(row, column)
                    )
                self.buttons[(row_number, column_number)] = button
                button_list.append(button)
        columns = len(self.board[0])
        grid = widgets.GridBox(
            button_list,
            layout=widgets.Layout(
                grid_template_columns=f"repeat({columns}, 64px)",
                grid_gap="6px",
            ),
        )
        self.grid_holder.children = (grid,)

    def _powered_squares(self):
        source = None
        for row_number, row in enumerate(self.board):
            for column_number, square in enumerate(row):
                if square["kind"] == B:
                    source = (row_number, column_number)
        powered = set()
        if source is None:
            return powered
        queue = [source]
        powered.add(source)
        while queue:
            row_number, column_number = queue.pop(0)
            bits = self.board[row_number][column_number]["bits"]
            for direction in DIRECTIONS:
                if not bits & direction:
                    continue
                row_step, column_step = STEP[direction]
                neighbour = (row_number + row_step, column_number + column_step)
                neighbour_row, neighbour_column = neighbour
                if not (0 <= neighbour_row < len(self.board)):
                    continue
                if not (0 <= neighbour_column < len(self.board[0])):
                    continue
                neighbour_bits = self.board[neighbour_row][neighbour_column]["bits"]
                if neighbour_bits & OPPOSITE[direction] and neighbour not in powered:
                    powered.add(neighbour)
                    queue.append(neighbour)
        return powered

    def _won(self, powered):
        lamps = {
            (row_number, column_number)
            for row_number, row in enumerate(self.board)
            for column_number, square in enumerate(row)
            if square["kind"] == L
        }
        return bool(lamps) and lamps.issubset(powered)

    def turn(self, row_number, column_number):
        square = self.board[row_number][column_number]
        if square["kind"] != X and not square["locked"]:
            square["bits"] = rotate(square["bits"])
            self.moves += 1
            self.refresh()

    def refresh(self):
        level = LEVELS[self.level_index]
        powered = self._powered_squares()
        won = self._won(powered)
        self.header.value = (
            f"<h2 style='margin:0;color:#35bde0'>⚡ {TEAM_NAME}: Level {self.level_index + 1}/3</h2>"
            f"<h3 style='margin:4px 0'>{level['title']}</h3>"
        )
        self.instructions.value = f"<p style='font-size:16px'>{level['message']} Click a wire to rotate it.</p>"
        for position, button in self.buttons.items():
            row_number, column_number = position
            square = self.board[row_number][column_number]
            kind = square["kind"]
            if kind == B:
                button.description = "🔋"
            elif kind == L:
                button.description = "💡" if position in powered else "○"
            else:
                button.description = GLYPH.get(square["bits"], "?")
            if kind == X:
                button.style.button_color = BOARD_COLOUR
            elif position in powered and kind == L:
                button.style.button_color = LAMP_COLOUR
            elif position in powered:
                button.style.button_color = POWERED_COLOUR
            else:
                button.style.button_color = WIRE_COLOUR
        if won:
            if self.level_index == len(LEVELS) - 1:
                self.status.value = (
                    f"<h3 style='color:#43a047'>🎉 {TEAM_NAME}: QUEST COMPLETE in {self.moves} moves! "
                    "Both lamps have power.</h3>"
                )
                self.next_button.description = "Quest complete"
                self.next_button.disabled = True
            else:
                self.status.value = (
                    f"<h3 style='color:#43a047'>✅ LEVEL COMPLETE in {self.moves} moves!</h3>"
                )
                self.next_button.description = "Next level"
                self.next_button.disabled = False
        else:
            self.status.value = f"<p><b>Moves:</b> {self.moves} &nbsp; Lamp off. Keep rotating.</p>"
            self.next_button.description = "Next level"
            self.next_button.disabled = True

    def _reset_clicked(self, unused):
        self.load_level(self.level_index)

    def _hint_clicked(self, unused):
        for row_number, row in enumerate(self.board):
            for column_number, square in enumerate(row):
                if not square["locked"] and square["kind"] != X:
                    if square["bits"] != square["solved"]:
                        self.status.value = (
                            f"<p style='color:#f59e0b'><b>Hint:</b> Try row {row_number + 1}, "
                            f"column {column_number + 1}.</p>"
                        )
                        return
        self.status.value = "<p>Hint: trace the glowing path from the battery.</p>"

    def _next_clicked(self, unused):
        if self.level_index < len(LEVELS) - 1:
            self.load_level(self.level_index + 1)


## 2. Make it yours

Change the team name and **one** colour. Keep the quotation marks. Then run the cell.

Try a colour such as `hotpink`, `orange`, `violet`, `gold`, `cyan`, or `lime`.


In [ ]:
# Change this line.
TEAM_NAME = 'Circuit Crew'
# Change ONE of these colour lines.
WIRE_COLOUR = 'deepskyblue'
POWERED_COLOUR = 'lime'

print(TEAM_NAME, 'is ready.')
print('Wire colour:', WIRE_COLOUR)


## 3. Play

Run the next cell. Your team name and colours appear in the game.

1. Click a blue wire tile to rotate it.
2. Trace the path from the battery. Connected tiles glow green.
3. Light every lamp.
4. Use **Hint** or **Reset level** whenever you need them.
5. Finish all three levels.


In [ ]:
quest = CircuitQuest()
quest.show()
print('Game ready. Click the wire tiles above.')


## 4. Next: the Life Checkpoint

Circuit Quest is the warm-up. **Do not build this tile board in Tinkercad.**

Next, open the [Life Checkpoint Upgrade](https://andrewandrade.ca/tech-edu-resources/tej3-4/computational-engineering/midterm-circuit-investigation.html). Correct the checkpoint in Jupyter, then build the checkpoint's two-loop circuit in Tinkercad. That circuit uses the $12\,\mathrm{V}$ and $5\,\mathrm{V}$ sources with the $5\,\mathrm{k\Omega}$, $7\,\mathrm{k\Omega}$, and $2\,\mathrm{k\Omega}$ resistors.


## Common errors

- No board appears: run the setup cell, your team-and-colours cell, and the play cell in order.
- Buttons show as text: confirm that **Python (Pyodide)** is the selected kernel.
- A lamp stays off: follow the green path and rotate the first blue tile after it.
- The game seems stuck: click **Hint** or **Reset level**.
- The notebook was reopened: restart the kernel and run both game cells again.


## Help improve this lesson (optional)

If something got in the way, leave a short note for the teacher:

1. Which step or word was unclear?
2. What do you still not know how to do?
3. What one change would make this easier for the next student?

If you can fix the wording, example, diagram, screenshot, or starter code, show the teacher. Useful fixes may be added to the course materials and credited. This feedback is optional.
